**Aprendizaje de Máquina Aplicado**

**Estudiantes:**
- Patricia Arango
- Santiago Higuita
- Alexander Pelaez

Repositorio del Proyecto: [Github](https://github.com/AlexanderPelaezJimenez/ml_project_eafit/tree/main)
___

**Universidad EAFIT**       
**Fecha:** 2026

---
## Tabla de Contenidos

1. **Introducción y objetivos**
2. **Setup reproducible**
3. **Carga del dataset limpio**
4. **Split estratificado por periodo**
5. **Feature engineering determinístico**
   - 5.1 Edad aproximada del estudiante
   - 5.2 Unificación de personas y cuartos del hogar
   - 5.3 Capital tecnológico
   - 5.4 Casteo del target a numérico
6. **Pipeline de preprocesamiento**
   - 6.1 Clasificación de columnas por tipo de encoding
   - 6.2 ColumnTransformer con sub-pipelines
   - 6.3 Encoding ordinal con orden explícito
7. **Definición de modelos candidatos**
   - 7.1 Familia lineal: Ridge
   - 7.2 Familia gradient boosting: LightGBM, XGBoost, CatBoost
8. **Hyperparameter tuning con Optuna**
   - 8.1 Función objetivo y grid de hiperparámetros
   - 8.2 Cross-validation 3-fold como protocolo de validación
   - 8.3 Optimización paralela aprovechando el M4
9. **Resultados y comparación de modelos**
   - 9.1 Mejor RMSE por modelo
   - 9.2 Mejores hiperparámetros encontrados
10. **Conclusión interim y próximos pasos**

---
## Introducción y objetivos

En este notebook entramos al modelado del proyecto. Hasta ahora teníamos un dataset limpio (3.4M registros, 27 features) y un baseline lineal de la entrega anterior. La pregunta concreta que respondemos aquí es: ¿qué tan bien podemos predecir el `punt_global` del Saber 11 a partir de las condiciones socioeconómicas y características del colegio, antes de que el estudiante presente el examen?

Para responderla, comparamos dos familias de modelos sobre la misma base reproducible:

- una familia lineal con regularización (Ridge), que sirve de referencia clara y fácil de interpretar,
- y tres gradient boosters (LightGBM, XGBoost, CatBoost), que son los que la literatura reporta como top performers en este tipo de problemas tabulares.

Todo pasa por un pipeline de scikit-learn con encoding por tipo de variable (ordinal, binario, one-hot, alta cardinalidad), envuelto en un `ColumnTransformer` que se re-fitea en cada fold del cross-validation. Eso garantiza que ninguna estadística aprendida (modas, categorías) cruce desde la validación al entrenamiento.

### Objetivos de la entrega

Al cerrar este notebook deberíamos haber cumplido con lo que pide la rúbrica:

1. Tener un **pipeline reproducible** end-to-end, desde el dataset limpio hasta el modelo entrenado.
2. **Comparar al menos dos familias** de modelos (lineal vs gradient boosting) con el mismo protocolo.
3. Definir un **protocolo de validación** sólido: cross-validation 3-fold sobre train, test holdout sin tocar.
4. Discutir **fairness/error por subgrupo** (el equivalente en regresión a la discusión de threshold/imbalance).
5. Cerrar con una **conclusión interim** que nombre el ganador, sus limitaciones y los próximos pasos.

### Decisiones clave que tomamos

- **Hyperparameter tuning con Optuna** (no grid search): mejor exploración del espacio, especialmente para los boosters con muchos hiperparámetros.
- **Split estratificado por periodo**, no temporal: cada cohorte (calendario A/B de cada año) está representada proporcionalmente en train y test.
- **Sin leakage por construcción**: el preprocessor vive dentro del pipeline, así que cada fold de validación nunca ve nada del fold de entrenamiento de ese mismo split.

### Regla que seguimos

> Primero limpiamos.
> Luego splitteamos.
> Después transformamos solo con lo que vio el train.
> Y solo entonces medimos en validación.


## Carga de Datos

In [1]:
from pathlib import Path

import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

import polars as pl
from loguru import logger

from sklearn.model_selection import train_test_split

SEED = 42

In [2]:
DATA_DIR = Path("../data/processed")

def load_processed_data(folder: str, format: str = "parquet"):

    logger.info(f'Cargando datos desde {folder} con formato {format}...')
    return pl.scan_parquet(f"{folder}/*.{format}").collect()

In [3]:
cleaned_dataset = load_processed_data(DATA_DIR, format="parquet")
cleaned_dataset.head(5).show()

2026-05-14 15:44:12.815 | INFO     | __main__:load_processed_data:5 - Cargando datos desde ../data/processed con formato parquet...


cole_sede_principal,cole_caracter,fami_cuartoshogar,fami_educacionpadre,estu_tipodocumento,estu_fechanacimiento,fami_tienecomputador,estu_genero,fami_tienelavadora,fami_personashogar,estu_depto_presentacion,estu_depto_reside,cole_naturaleza,cole_calendario,estu_mcpio_reside,periodo,cole_genero,fami_estratovivienda,cole_area_ubicacion,cole_bilingue,cole_mcpio_ubicacion,estu_mcpio_presentacion,fami_educacionmadre,fami_tieneautomovil,cole_jornada,punt_global,fami_tieneinternet,cole_depto_ubicacion
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""N""","""TÉCNICO/ACADÉMICO""","""Dos""","""Primaria completa""","""TI""","""15/02/2003""","""Si""","""F""","""Si""","""1 a 2""","""HUILA""","""HUILA""","""OFICIAL""","""A""","""AIPE""","""20194""","""MIXTO""","""Estrato 2""","""RURAL""","""N""","""AIPE""","""AIPE""","""Postgrado""","""No""","""COMPLETA""","""339""","""Si""","""HUILA"""
"""S""","""TÉCNICO/ACADÉMICO""","""Dos""","""Primaria incompleta""","""TI""","""08/03/2002""","""No""","""F""","""No""","""5 a 6""","""HUILA""","""HUILA""","""OFICIAL""","""A""","""LA PLATA""","""20194""","""MIXTO""","""Estrato 1""","""URBANO""","""N""","""LA PLATA""","""LA PLATA""","""Primaria incompleta""","""No""","""COMPLETA""","""199""","""No""","""HUILA"""
"""S""","""TÉCNICO/ACADÉMICO""","""Cuatro""","""Secundaria (Bachillerato) comp…","""TI""","""27/07/2000""","""Si""","""M""","""Si""","""Cinco""","""VALLE""","""VALLE""","""OFICIAL""","""A""","""CALI""","""20162""","""MIXTO""","""Estrato 5""","""URBANO""","""N""","""CALI""","""CALI""","""Secundaria (Bachillerato) inco…","""Si""","""MAÑANA""","""272""","""Si""","""VALLE"""
"""S""","""TÉCNICO/ACADÉMICO""","""Cinco""","""Primaria incompleta""","""TI""","""07/12/1999""","""No""","""F""","""Si""","""5 a 6""","""ANTIOQUIA""","""ANTIOQUIA""","""NO OFICIAL""","""A""","""MEDELLÍN""","""20172""","""MIXTO""","""Estrato 2""","""URBANO""","""N""","""MEDELLIN""","""ITAGÜÍ""","""Secundaria (Bachillerato) inco…","""No""","""SABATINA""","""253""","""Si""","""ANTIOQUIA"""
"""S""","""TÉCNICO/ACADÉMICO""","""Dos""","""Primaria incompleta""","""CC""","""15/09/1995""","""No""","""M""","""No""","""Cuatro""","""TOLIMA""","""TOLIMA""","""OFICIAL""","""A""","""SAN LUIS""","""20142""","""MIXTO""","""Estrato 1""","""RURAL""","""N""","""SAN LUIS""","""GUAMO""","""Primaria incompleta""","""No""","""MAÑANA""","""212""","""No""","""TOLIMA"""


## Split estratificado por Período

In [5]:
print(cleaned_dataset['periodo'].value_counts().sort(by='periodo').head(10))

shape: (10, 2)
┌─────────┬────────┐
│ periodo ┆ count  │
│ ---     ┆ ---    │
│ str     ┆ u32    │
╞═════════╪════════╡
│ 20142   ┆ 544535 │
│ 20151   ┆ 25947  │
│ 20152   ┆ 542450 │
│ 20161   ┆ 13064  │
│ 20162   ┆ 548206 │
│ 20171   ┆ 13018  │
│ 20172   ┆ 546261 │
│ 20181   ┆ 19798  │
│ 20191   ┆ 12540  │
│ 20194   ┆ 546211 │
└─────────┴────────┘


In [6]:
X = cleaned_dataset.drop("punt_global")
y = cleaned_dataset["punt_global"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=cleaned_dataset["periodo"]
)

print(f"Train: {X_train.shape[0]:,} filas | Test: {X_test.shape[0]:,} filas")
print(f"Proporcion test: {X_test.shape[0] / cleaned_dataset.shape[0] * 100:.1f}%")

Train: 2,716,667 filas | Test: 679,167 filas
Proporcion test: 20.0%


## Pipeline de preprocesamiento

In [ ]:
import numpy as np
import pandas as pd

import optuna
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge, Lasso, LinearRegression

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

/Users/santiago.higuitau/Documents/Machine_Learning/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
def regression_report(y_true: pd.Series, y_pred: np.ndarray, mod_tag: str = "default") -> dict:

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "model-experiment": mod_tag,
        "r2": r2,
        "rmse": rmse,
        "mae": mae,
    }


def evaluate_model(
    estimator,
    X_train,
    y_train,
    X_test,
    y_test,
    mod_tag: str
) -> dict:

    estimator.fit(X_train, y_train)
    predictions = estimator.predict(X_test)
    
    report = regression_report(y_test, predictions, mod_tag)
    logger.info(f"Evaluación del modelo {mod_tag}: {report}")

    return report


def deterministic_processor(
    X: pl.DataFrame,
    mapeo_personas: dict,
    mapeo_cuartos: dict
) -> pl.DataFrame:

    X_calculated = (
        X.clone()
        .with_columns(
        (
            pl.col("periodo").str.slice(0, 4).cast(pl.Int32) - 
            pl.col("estu_fechanacimiento").str.slice(6, 4).cast(pl.Int32)
        )
        .alias("edad_estudiante_aprox")
        )
        .with_columns(
            pl.col('edad_estudiante_aprox')
            .clip(15, 60)
            .alias('edad_estudiante_aprox')
        )
        .with_columns(
            pl.col("fami_personashogar")
            .replace(mapeo_personas)
            .cast(pl.Int32, strict=False)
            .alias("fami_personashogar_num"),

            pl.col("fami_cuartoshogar")
            .replace(mapeo_cuartos)
            .cast(pl.Int32, strict=False)
            .alias("fami_cuartos_num")
        )
        .drop([
            "fami_personashogar",
            "fami_cuartoshogar",
            "estu_fechanacimiento"
        ])
        .with_columns(
            ((pl.col("fami_tieneinternet") == "Si").cast(pl.Int8) + 
            (pl.col("fami_tienecomputador") == "Si").cast(pl.Int8))
            .alias("capital_tecnologico")
        )
    )

    return X_calculated

In [9]:
mapeo_personas = {
    "Uno": 1,
    "Dos": 2,
    "Tres": 3,
    "Cuatro": 4,
    "Cinco": 5,
    "Seis": 6,
    "Siete": 7,
    "Ocho": 8,
    "Nueve": 9,
    "Diez": 10,
    "Once": 11,
    "Doce o más": 12,
    "1 a 2": 1,
    "3 a 4": 3,
    "5 a 6": 5,
    "7 a 8": 7,
    "9 o más": 9
}


mapeo_cuartos = {
    "Uno": 1,
    "Dos": 2,
    "Tres": 3,
    "Cuatro": 4,
    "Cinco": 5,
    "Seis": 6,
    "Seis o mas": 6,
    "Siete": 7,
    "Ocho": 8,
    "Nueve": 9,
    "Diez o más": 10,
}

In [ ]:
# Apply deterministic processor
X_train = deterministic_processor(X_train.clone(), mapeo_personas, mapeo_cuartos)
X_test = deterministic_processor(X_test.clone(), mapeo_personas, mapeo_cuartos)

X_train_pd = X_train.to_pandas()
X_test_pd = X_test.to_pandas()

# Castear target a float32 (XGBoost requiere numerico, no strings)
y_train_np = y_train.cast(pl.Int32).to_numpy().astype(np.float32)
y_test_np = y_test.cast(pl.Int32).to_numpy().astype(np.float32)

In [11]:
# Definir orden de los estratos:
orden_estrato = [
    'Sin Estrato',
    'Estrato 1',
    'Estrato 2',
    'Estrato 3',
    'Estrato 4',
    'Estrato 5',
    'Estrato 6'
]

# Definir orden de la educación:
orden_educacion = [
    'Ninguno',
    'No sabe',
    "No Aplica",
    'Primaria incompleta',
    'Primaria completa',
    'Secundaria (Bachillerato) incompleta',
    'Secundaria (Bachillerato) completa',
    'Técnica o tecnológica incompleta',
    'Técnica o tecnológica completa',
    'Educación profesional incompleta',
    'Educación profesional completa',
    'Postgrado',
]

# Clasificación de las variables:
ordinal_cols = [
    'fami_estratovivienda',
    'fami_educacionmadre',
    'fami_educacionpadre'
]

binary_cols = [
    'fami_tieneinternet',
    'fami_tienecomputador',
    'fami_tieneautomovil',
    'fami_tienelavadora',
    'cole_bilingue',
    'cole_sede_principal',
    'estu_genero',
]

nominal_low_cols = [
    'cole_naturaleza',
    'cole_area_ubicacion',
    'cole_calendario',
    'cole_jornada',
    'cole_genero',
    'cole_caracter',
    'estu_tipodocumento',
    'estu_depto_reside',
    'estu_depto_presentacion',
    'cole_depto_ubicacion',
    'periodo',
]

nominal_high_cols = [
    'estu_mcpio_reside',
    'cole_mcpio_ubicacion',
    'estu_mcpio_presentacion',
]

numeric_cols = [
    'edad_estudiante_aprox',
    'fami_personashogar_num',
    'fami_cuartos_num',
    "capital_tecnologico",
]

# Sub-pipelines
preprocessor = ColumnTransformer([
    ("ordinal", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(
            categories=[orden_estrato, orden_educacion, orden_educacion],
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )),
    ]), ordinal_cols),

    ("binary", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), binary_cols),

    ("nominal_low", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), nominal_low_cols),

    ("nominal_high", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), nominal_high_cols),

    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ]), numeric_cols),
], remainder="drop")

## Definición de modelos candidatos

In [ ]:
from optuna import Trial, create_study, logging

logging.set_verbosity(logging.WARNING)

N_TRIALS_LINEAR = 5    # Ridge/Lasso: solo 1 param, converge rápido
N_TRIALS_TREE = 12     # Árboles: más params, necesitan más exploración
CV_FOLDS = 3           # Tuning inicial
CV_FOLDS_FINAL = 5     # Evaluación final del ganador por modelo
PARALLEL_TRIALS = 2    # Trials en paralelo para árboles
CORES_PER_MODEL = 5    # 10 cores / 2 trials paralelos

In [13]:
def build_model_grid(trial: Trial, model_selected: str) -> dict:

    if model_selected == "ridge":
        return {
            "model_type": "linear",
            "params_grid": {
                "alpha": trial.suggest_float("alpha", 1e-3, 1000.0, log=True),
            },
        }

    elif model_selected == "lasso":
        return {
            "model_type": "linear",
            "params_grid": {
                "alpha": trial.suggest_float("alpha", 1, 500.0, log=True),
                "max_iter": 2000,
            },
        }

    elif model_selected == "randomForest":
        return {
            "model_type": "tree",
            "params_grid": {
                "n_estimators": trial.suggest_int("n_estimators", 100, 500),
                "max_depth": trial.suggest_int("max_depth", 6, 20),
                "min_samples_split": trial.suggest_int("min_samples_split", 5, 50),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 30),
                "max_features": trial.suggest_float("max_features", 0.3, 1.0),
            },
        }

    elif model_selected == "lightGBM":
        return {
            "model_type": "tree",
            "params_grid": {
                "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
                "max_depth": trial.suggest_int("max_depth", 4, 12),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "num_leaves": trial.suggest_int("num_leaves", 20, 150),
                "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            },
        }

    elif model_selected == "xgboost":
        return {
            "model_type": "tree",
            "params_grid": {
                "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
                "max_depth": trial.suggest_int("max_depth", 4, 12),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 50),
                "gamma": trial.suggest_float("gamma", 1e-8, 5.0, log=True),
            },
        }

    elif model_selected == "catboost":
        return {
            "model_type": "tree",
            "params_grid": {
                "iterations": trial.suggest_int("iterations", 200, 1000),
                "depth": trial.suggest_int("depth", 4, 10),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
                "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
                "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            },
        }

    return {}

In [ ]:
def objective_model(trial, model_selected: str, random_seed: int):

    config = build_model_grid(trial, model_selected)
    params = config.get("params_grid", {})
    model_type = config.get("model_type", None)

    assert params, f"Modelo {model_selected} sin parámetros en la configuración."
    assert model_type is not None, f"Modelo {model_selected} no encontrado en la configuración."


    if model_type == 'linear':

        model = Ridge(**params) if model_selected == 'ridge' else Lasso(**params)

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("scaler", StandardScaler()),
            ("model", model),
        ])


    elif model_type == 'tree':

        if model_selected == 'randomForest':
            model = RandomForestRegressor(**params, random_state=random_seed, n_jobs=CORES_PER_MODEL)

        elif model_selected == 'lightGBM':
            model = LGBMRegressor(**params, random_state=random_seed, verbose=-1, n_jobs=CORES_PER_MODEL)

        elif model_selected == 'xgboost':
            model = XGBRegressor(**params, random_state=random_seed, verbosity=0, tree_method="hist", nthread=CORES_PER_MODEL)

        else:
            model = CatBoostRegressor(**params, random_state=random_seed, verbose=0, thread_count=CORES_PER_MODEL)

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model),
        ])

    scores = cross_val_score(
        pipeline,
        X_train_pd,
        y_train_np,
        cv=CV_FOLDS,
        scoring="neg_root_mean_squared_error",
        n_jobs=1
    )

    return -scores.mean()

## Hyperparameter tuning con Optuna

In [15]:
studies = dict()
models_to_tune = [
    'ridge',
    # 'lasso',
    # 'randomForest',
    'lightGBM',
    'xgboost',
    'catboost'
]

for model_name in models_to_tune:
    logger.info(f'Iniciando tuning usando Optuna para el modelo: {model_name}')

    # Trials y paralelismo según tipo de modelo
    is_linear = model_name in ['ridge', 'lasso']
    n_trials = N_TRIALS_LINEAR if is_linear else N_TRIALS_TREE
    n_jobs_optuna = 1 if is_linear else PARALLEL_TRIALS

    study = create_study(direction="minimize", study_name=f'Experimento usando: {model_name}')
    study.optimize(
        lambda trial, m=model_name, r=SEED: objective_model(trial, model_selected=m, random_seed=r),
        n_trials=n_trials,
        n_jobs=n_jobs_optuna,
        show_progress_bar=True
    )

    studies[model_name] = study

    logger.info(f'Mejor RMSE CV: {study.best_value:.4f}')
    logger.info(f'Mejores params: {study.best_params}')

2026-05-14 15:44:45.857 | INFO     | __main__:<module>:12 - Iniciando tuning usando Optuna para el modelo: ridge
Best trial: 4. Best value: 40.1011: 100%|██████████| 5/5 [04:53<00:00, 58.73s/it]
2026-05-14 15:49:39.542 | INFO     | __main__:<module>:29 - Mejor RMSE CV: 40.1011
2026-05-14 15:49:39.542 | INFO     | __main__:<module>:30 - Mejores params: {'alpha': 909.3215408054189}
2026-05-14 15:49:39.542 | INFO     | __main__:<module>:12 - Iniciando tuning usando Optuna para el modelo: lightGBM
Best trial: 4. Best value: 36.8257: 100%|██████████| 12/12 [26:20<00:00, 131.70s/it]
2026-05-14 16:15:59.993 | INFO     | __main__:<module>:29 - Mejor RMSE CV: 36.8257
2026-05-14 16:15:59.993 | INFO     | __main__:<module>:30 - Mejores params: {'n_estimators': 933, 'max_depth': 7, 'learning_rate': 0.16594025491273964, 'num_leaves': 150, 'min_child_samples': 77, 'subsample': 0.9760307243309925, 'colsample_bytree': 0.8044179801644085, 'reg_alpha': 0.0004455030463147903, 'reg_lambda': 7.627852420889

---
## Resultados y comparación de modelos

Después de correr Optuna sobre los cuatro modelos con cross-validation 3-fold sobre los 2.7M registros de train, estos son los resultados:

| Modelo | Mejor RMSE CV | Tiempo de tuning | Trials |
|--------|--------------:|-----------------:|-------:|
| **LightGBM** | **36.8257** | ~26 min | 12 |
| XGBoost | 36.8416 | ~59 min | 12 |
| CatBoost | 36.9321 | ~48 min | 12 |
| Ridge | 40.1011 | ~5 min | 5 |

### Mejor modelo: LightGBM

LightGBM ganó con un margen muy estrecho sobre XGBoost (0.016 puntos de RMSE) y CatBoost (0.107 puntos). Los hiperparámetros óptimos fueron:

```python
{
    "n_estimators": 933,
    "max_depth": 7,
    "learning_rate": 0.166,
    "num_leaves": 150,
    "min_child_samples": 77,
    "subsample": 0.976,
    "colsample_bytree": 0.804,
    "reg_alpha": 0.0004,
    "reg_lambda": 7.6e-06,
}


Algunas notas:

- El gap entre Ridge y los boosters es de ~3.3 puntos de RMSE. Esto confirma parcialmente que las relaciones entre features y target no son lineales, pues hay interacciones (ej. el efecto del estrato depende del tipo de colegio) y umbrales (ej. tener internet importa más en ciertos rangos de educación de los padres) que un modelo lineal no puede capturar.

- Los tres boosters quedan empatados dentro de 0.1 puntos. Esto sugiere que el cuello de botella ya no es el algoritmo, sino la información disponible en las features. Para mejorar más el RMSE vamos a invertir para la última fase en feature engineering (índice socioeconómico compuesto, valor agregado del colegio, agregaciones jerárquicas, etc) más que en probar modelos más complejos.

- Interpretación del RMSE: un error promedio de 36.83 puntos sobre una escala de 0-500 equivale a ~7.4% de la escala. Considerando que la desviación estándar del target es ~50, el modelo explica aproximadamente el 46% de la varianza del puntaje (R² implícito ≈ 0.46).

- Validez del protocolo: los números reportados son de cross-validation sobre train. El test holdout (679K registros) no se ha tocado y se reserva para la evaluación final del modelo elegido en la próxima iteración.

## Conclusión Etapa

**Análisis de error por subgrupo**

En regresión, el equivalente a la discusión de imbalance/threshold es verificar si el modelo 
es igualmente preciso para todos los grupos. Ampliaremos en la siguiente entrega, ejemplos como:

- RMSE por estrato (1 vs 6): ¿el modelo predice peor para estratos bajos?
- RMSE urbano vs rural
- RMSE por periodo: ¿hay concept drift?

Esto es crítico porque un modelo que subestima sistemáticamente a un grupo puede perpetuar desigualdades si se usa para asignar recursos.

### Conclusión interim

1. **LightGBM es el mejor modelo** (RMSE CV = 36.83), prácticamente empatado con XGBoost (36.84) 
   y CatBoost (~37.02). Ridge queda 3.3 puntos atrás (40.10).

2. **Las variables socioeconómicas tienen señal predictiva fuerte.** El modelo reduce el error 
   de ~50 (std del target) a ~37 puntos, explicando aproximadamente el 46% de la varianza.

3. **El gap entre lineal y boosting (~3.3 puntos) confirma no-linealidades e interacciones** 
   que Ridge no puede capturar. Por ejemplo, el efecto del estrato seguramente depende del tipo 
   de colegio y del acceso a tecnología, y los árboles modelan eso nativamente.

4. **El empate entre los tres boosters sugiere que el cuello de botella ya no es el modelo, 
   sino la información disponible.** Mejorar más el RMSE probablemente requiere mejor feature 
   engineering, no más complejidad de modelo.

5. **No hay leakage:** el pipeline completo (imputación + encoding + modelo) se re-fitea 
   en cada fold del CV. Los puntajes parciales (`punt_ingles`, `punt_matematicas`, etc.) 
   están excluidos por ser componentes directos del target.

6. **Limitaciones identificadas:**
   - El error promedio de 37 puntos es ~7.4% de la escala (0-500), aceptable pero mejorable
   - No medimos aún si el error es equitativo entre subgrupos (estrato, urbano/rural, etc)
   - El test holdout no se ha tocado; los números reportados son de cross-validation

7. **Próximos pasos para la entrega final:**
   - Evaluación en test holdout con el mejor modelo (LightGBM)
   - Feature engineering adicional: índice socioeconómico compuesto, valor agregado del colegio, entre otros
   - SHAP para interpretabilidad: ¿qué features dominan? ¿hay umbrales no lineales?, etc
   - Análisis de fairness: RMSE por estrato, urbano/rural, periodo, etc
   - Serialización del modelo + API REST con FastAPI
   - Documentación: model card describiendo el modelo y sus limitaciones